# 9. The cluster wall-scan: is the entanglement barrier truncation-independent?

This notebook analyzes the BSC cluster sweep that completes the ε-scan started locally in notebook
8. There, halving the bond dimension (χ=32 vs χ=64) left the wall exactly where it was — a
gap-limited, not resolution-limited, verdict. Two more ε knobs were still untested: the truncation
*algorithm* (`:rtm` vs `:rdm`) and a tighter cutoff. A workmate at BSC ran all three, at χ=64,
across T=2..14, using an independent SLURM array job per T (one node per time point, for speed).

That data arrived in two waves. The first wave (analyzed in §1-§3 below) was **cold-started**: each
T was computed independently, with no warm seed from the previous T. This is a genuinely different
setup from every local run in this thesis, and it shows: the entropy dome inflates at T≈6 instead
of T≈10, and the naive eigenvalue selector picks the wrong branch at a couple of T's. We diagnose
exactly why in §4, fix the selector (promoting the phase-based classifier from notebook 5 into
`src/transverse_tools.jl`), and use it to show that the *eigenvalue-level* physics — phase rigidity,
the RTM-vs-RDM comparison — was sound all along; only the *entropy*, which needs the individual
eigenvector, was compromised by the missing warm start. A corrected, warm-started rerun is now in
flight on the same cluster (§6), so this notebook is written to be rerun once that data lands too.

**Honesty box.** Everything about *eigenvalues* and *phase rigidity* in the cold data below is
trustworthy - those quantities are exactly as well-conditioned whether cold- or warm-started (an
eigenvalue is a Rayleigh quotient, quadratically insensitive to the eigenvector error; see notebook
8, section "why the eigenvector route fails"). Everything about the *entropy dome* and *where the
wall sits* is **not** directly comparable to the local warm-started baseline until the rerun lands -
that comparison is exactly what §4 is about.

In [ ]:
# Setup: the shared thesis library and this notebook's constants.
include("../src/thesislib.jl")
using JLD2, Printf, Plots
gr()

const P_NNN  = 0.1
const NBETA  = 4

# The three cold-array directories the workmate's SLURM pipeline wrote into.
const COLD_DIRS = Dict(
    "rtm64_full" => "../cluster/rtm_array",
    "rdm64"      => "../cluster/rdm_array",
    "cut_tight"  => "../cluster/cutoff_array",
)

println("thesislib + phase-classification toolkit loaded (classify_tower, tower_gap, ",
        "pick_phys_continuity, block_transfer_eigs_adaptive -- see src/transverse_tools.jl).")

## Section 1 - Collect: merge the cold array results into one organized cache

Each cold-array T-point landed as its own file, `cluster/<mode>_array/worker_results_T<T>.jld2`.
This cell merges all three configurations into a single, organized
`results/data/cluster/cold_sweep.jld2`, keyed by `(label, T)` exactly like every other cache in this
thesis. It is idempotent and safe to rerun any time more points land from the cluster.

In [ ]:
# Section 1: scan the raw per-T files and merge them into one organized cache.
mkpath("../results/data/cluster")
cold_cachefile = "../results/data/cluster/cold_sweep.jld2"

cold = Dict{Tuple{String,Float64},Any}()
for (label, dir) in COLD_DIRS
    isdir(dir) || continue
    for filename in readdir(dir)
        regex_match = match(r"worker_results_T([\d.]+)\.jld2", filename)
        regex_match === nothing && continue

        T = parse(Float64, regex_match.captures[1])
        per_T_data = load(joinpath(dir, filename), "done")
        key = (label, T)
        if haskey(per_T_data, key)
            cold[key] = per_T_data[key]
        end
    end
end

jldsave(cold_cachefile; done=cold)

println("merged cache: ", cold_cachefile)
for label in ["rtm64_full", "rdm64", "cut_tight"]
    n_points = count(k -> k[1] == label, keys(cold))
    println("  ", label, ": ", n_points, " points")
end

## Section 2 - Headline: the wall is truncation-independent

The single most important comparison in this notebook. Phase rigidity - the direct measure of how
well-defined the individual eigenvector is - plotted for all three cold cluster configurations
(`:rtm`, `:rdm`, tighter cutoff) alongside the two local, warm-started chi-scan series from notebook
8 (chi=32 and chi=64). If the near-exceptional point is a property of the *physics* (the closing
transfer gap) rather than of any particular numerical knob, all five series should collapse onto the
same geometric decay, regardless of bond dimension, cutoff, or truncation algorithm.

In [ ]:
# Section 2: phase rigidity vs T, five series overlaid.
local_chi64 = load("../results/data/nb13_eigvec_ladder.jld2", "done")
local_chi32_all = load("../results/data/nb13_wallscan.jld2", "done")
local_chi32 = Dict(T => local_chi32_all[("chi32", T)] for (lbl, T) in keys(local_chi32_all) if lbl == "chi32")

rigidity_plot = plot(
    xlabel="T", ylabel="phase rigidity  r_j  (member 1)", yscale=:log10,
    title="Phase rigidity: 5 series, one physics", legend=:bottomleft, framestyle=:box,
)

# local, warm-started series (solid lines, filled markers)
for (label, data, color) in [("local chi=64 (warm)", local_chi64, :black), ("local chi=32 (warm)", local_chi32, :gray)]
    Ts_here = sort(collect(keys(data)))
    rigidity_here = [data[T].rigidity[1] for T in Ts_here]
    plot!(rigidity_plot, Ts_here, rigidity_here, marker=:circle, lw=2, color=color, label=label)
end

# cluster, cold-started series (dashed lines, open markers)
cluster_colors = Dict("rtm64_full" => :red, "rdm64" => :blue, "cut_tight" => :green)
for label in ["rtm64_full", "rdm64", "cut_tight"]
    Ts_here = sort([T for (lbl, T) in keys(cold) if lbl == label && !haskey(cold[(lbl, T)], :error)])
    isempty(Ts_here) && continue
    rigidity_here = [cold[(label, T)].rigidity[1] for T in Ts_here]
    plot!(rigidity_plot, Ts_here, rigidity_here, marker=:diamond, ls=:dash, lw=2,
          color=cluster_colors[label], label="cluster " * label * " (cold)")
end

hline!(rigidity_plot, [1.0], ls=:dot, color=:gray, label="")

mkpath("../results/imgs")
savefig(rigidity_plot, "../results/imgs/nb9_rigidity_universal.png")
rigidity_plot

If the plot above shows all five series tracking the same geometric collapse, that is the
headline of this notebook: **the near-exceptional point does not care about bond dimension, cutoff,
or truncation algorithm.** It is a property of the transfer matrix's closing gap, full stop - the
last of the three epsilon-dials (truncation algorithm) is now closed alongside bond dimension
(notebook 8) and joins the same verdict.

## Section 3 - RTM vs RDM: does the truncation algorithm matter?

Two questions, answered independently. First: do `:rtm` and `:rdm` agree on the physics (the
eigenvalue and its conditioning)? Second: which is cheaper to run? The eigenvalue-agreement check
below re-derives the physical branch with `pick_phys_continuity` and `classify_tower` from the
*stored* spectra - not the naive rank-based `i0` each worker.jl computed at runtime - so this
comparison does not inherit any selector mistakes.

In [ ]:
# Section 3a: eigenvalue agreement across rtm / rdm / cutoff at every shared T.
function reselect(entry, previous_phys)
    theta = entry.theta
    i0 = pick_phys_continuity(theta, previous_phys)
    return theta[i0], i0
end

shared_Ts = sort(collect(intersect(
    Set(T for (lbl, T) in keys(cold) if lbl == "rtm64_full" && !haskey(cold[(lbl, T)], :error)),
    Set(T for (lbl, T) in keys(cold) if lbl == "rdm64" && !haskey(cold[(lbl, T)], :error)),
    Set(T for (lbl, T) in keys(cold) if lbl == "cut_tight" && !haskey(cold[(lbl, T)], :error)),
)))

println("Reselected |theta_phys| (rank-based selector at runtime was NOT trusted here):")
@printf("%-5s  %-12s  %-12s  %-12s\n", "T", "rtm", "rdm", "cutoff")
prev = Dict("rtm64_full" => nothing, "rdm64" => nothing, "cut_tight" => nothing)
for T in shared_Ts
    vals = String[]
    for label in ["rtm64_full", "rdm64", "cut_tight"]
        theta_phys, i0 = reselect(cold[(label, T)], prev[label])
        prev[label] = theta_phys
        push!(vals, @sprintf("%.5f", abs(theta_phys)))
    end
    @printf("%-5.1f  %-12s  %-12s  %-12s\n", T, vals[1], vals[2], vals[3])
end

In [ ]:
# Section 3b: cost comparison - elapsed time and iteration count per T.
@printf("%-5s  %-18s  %-18s  %-18s\n", "T", "rtm (s, iters)", "rdm (s, iters)", "cutoff (s, iters)")
function fmt_cost(label, T)
    haskey(cold, (label, T)) || return "--"
    entry = cold[(label, T)]
    haskey(entry, :error) && return "ERROR"
    return @sprintf("%7.0f  @%-5d", entry.elapsed, entry.niters)
end
all_Ts = sort(unique(T for (lbl, T) in keys(cold)))
for T in all_Ts
    @printf("%-5.1f  %-18s  %-18s  %-18s\n", T, fmt_cost("rtm64_full", T), fmt_cost("rdm64", T), fmt_cost("cut_tight", T))
end

total_time(label) = sum(cold[(label, T)].elapsed for T in all_Ts if haskey(cold, (label, T)) && !haskey(cold[(label, T)], :error))
rtm_total, rdm_total, cutoff_total = total_time("rtm64_full"), total_time("rdm64"), total_time("cut_tight")
println()
@printf("Totals: rtm = %.1fh   rdm = %.1fh   cutoff = %.1fh\n", rtm_total/3600, rdm_total/3600, cutoff_total/3600)

shared9 = [T for T in all_Ts if T <= 9.0]
rtm_9 = sum(cold[("rtm64_full", T)].elapsed for T in shared9)
rdm_9 = sum(cold[("rdm64", T)].elapsed for T in shared9)
@printf("T=2..9 only: rtm = %.1fh, rdm = %.1fh  -> rdm is %.1fx slower than rtm\n",
        rtm_9/3600, rdm_9/3600, rdm_9/rtm_9)

**Verdict: stick with RTM.** The eigenvalues agree to 4-5 digits between `:rtm` and `:rdm` at
every T (Section 3a) - both algorithms see the same physics. But `:rdm` is roughly 4x slower than
`:rtm` over the shared T=2..9 range (and the gap widens further at larger T, where `:rdm` was
observed taking upwards of 10x longer per point), despite frequently reporting `"converged"` where
`:rtm` reports `"stuck"` - the fewer-iterations outcome comes at a far higher cost *per* iteration
(independent, Hermitian truncation of each vector, rather than the cheaper joint RTM truncation),
so it does not translate into a net time saving. RDM's theoretical advantage - staying
well-conditioned through the near-degeneracy - does not show up as a *practical* one: Section 2
already showed its phase rigidity collapses at the identical rate. There is no dial here that buys
back reach; there is only a slower way to hit the same wall. Production stays RTM.

## Section 4 - The warm-start lesson: why the cold dome breaks four steps early

Two anomalies in the cold data, both traced to the same root cause: every T in the cold array was
computed by an independent SLURM task with `seedL=nothing, seedR=nothing` and no continuity anchor
(`previous_physical_value` was always `nothing`, so `pick_phys` fell back to naively picking the
top-2 eigenvalues by modulus, every single time).

In [ ]:
# Section 4a: dome peak vs T, cold cluster (rtm) vs local warm-started chi=64.
dome_plot = plot(xlabel="T", ylabel="peak Re(S_2)", title="Where does the dome break?",
                  legend=:topleft, framestyle=:box)

Ts_local = sort(collect(keys(local_chi64)))
peaks_local = [local_chi64[T].peak for T in Ts_local]
plot!(dome_plot, Ts_local, peaks_local, marker=:circle, lw=2, color=:black, label="local chi=64 (warm)")

Ts_cold_rtm = sort([T for (lbl, T) in keys(cold) if lbl == "rtm64_full" && !haskey(cold[(lbl, T)], :error)])
peaks_cold_rtm = [cold[("rtm64_full", T)].peak for T in Ts_cold_rtm]
plot!(dome_plot, Ts_cold_rtm, peaks_cold_rtm, marker=:diamond, ls=:dash, lw=2, color=:red,
      label="cluster rtm (cold)")

dome_plot

The cold dome inflates around T≈6; the local warm-started baseline stays clean to T≈9-10 at
the *same* phase rigidity (Section 2 already showed rigidity is identical between the two). The
difference is not numerical conditioning - it is which branch the iteration lands on. Near the
near-exceptional point, a cold random restart has nothing biasing it toward the physically
continuous eigenvector; a warm-started iteration starts *already close* to the right answer, which
matters increasingly as the gap closes and the basin around the physical branch shrinks.

**The second anomaly - wrong-branch selection at T=12/13.** The cold `rtm` array's naive selector
(`pick_phys` with `previous_physical_value=nothing` on every call) always default to the two
largest-modulus eigenvalues. That is fine while the physical branch IS the largest modulus one, but
breaks down once other cluster members grow comparable in size.

In [ ]:
# Section 4b: the T=12/13 branch-hop, visible directly in the raw stored spectra.
for T in [11.0, 12.0, 13.0, 14.0]
    haskey(cold, ("rtm64_full", T)) || continue
    entry = cold[("rtm64_full", T)]
    haskey(entry, :error) && continue
    naive_i0 = 1   # the naive selector: top-of-block by modulus, no continuity
    println("T=", T, "  naive |theta_phys|=", round(abs(entry.theta[naive_i0]), digits=4),
            "   full block moduli=", round.(abs.(entry.theta), digits=4))
end

**Answering the resumability question directly.** Warm chaining buys two separate things, and
it is worth being precise about which one the cold array was missing:

1. **Seeding** - handing the iteration the previous T's converged blocks as a starting point. Purely
   a speed/robustness aid near the wall; lost on any restart unless the blocks are checkpointed to
   disk.
2. **Continuity anchoring** - remembering the previous T's accepted eigenvalue so the *selector*
   (`pick_phys_continuity`) can pick the closest-by-value branch instead of guessing by rank. This
   is what actually prevents the T=12/13 branch hops above.

The cold array had *neither*. So "the computation cannot stop and resume later" was true of the
code as it stood, but not for a fundamental reason - it was simply never taught how to persist
either piece of warm-start information across a restart. The corrected driver
(`cluster/wall_scan_cluster.jl`) now checkpoints the converged L/R blocks to disk after every T
and reloads them on startup, so a SLURM job that hits its walltime cap and gets resubmitted resumes
truly warm - not just continuity-anchored - exactly like an uninterrupted run would.

## Section 5 - Applying the corrected selector to the cold data

`classify_tower` and `pick_phys_continuity` (promoted into `src/transverse_tools.jl` this session,
after the same naive-selector bug was independently found in notebook 5's p-sweep) let us
re-examine the cold spectra without trusting the in-kernel `i0`. The table below re-derives
`theta_phys` for every T in the cold `rtm` array using full-block continuity tracking, and flags
every T where that disagrees with the naive top-of-block choice.

In [ ]:
# Section 5: re-derive theta_phys for the whole cold rtm ladder, flag every disagreement.
println("Re-selection over the full cold rtm64_full ladder:")
@printf("%-5s  %-10s  %-10s  %-10s\n", "T", "naive", "corrected", "agree?")
prev_corrected = nothing
for T in sort([T for (lbl, T) in keys(cold) if lbl == "rtm64_full" && !haskey(cold[(lbl, T)], :error)])
    entry = cold[("rtm64_full", T)]
    naive_val = abs(entry.theta[1])
    i0_corrected = pick_phys_continuity(entry.theta, prev_corrected)
    corrected_val = entry.theta[i0_corrected]
    prev_corrected = corrected_val
    agree = i0_corrected == 1 ? "yes" : "NO -- branch hop"
    @printf("%-5.1f  %-10.4f  %-10.4f  %-10s\n", T, naive_val, abs(corrected_val), agree)
end

This is exactly the same lesson NB5's p-sweep independently surfaced: a rank-based, modulus-only
selector is only safe while the physical branch happens to be the largest member of the block. Once
a near-degenerate cluster forms - whether triggered by NNN frustration at larger p (NB5) or simply
by getting deep enough into a cold-started T-ladder (here) - the selector must track the *value*,
not the *rank*. That is now a `src/`-level tool (`pick_phys_continuity`, `classify_tower`,
`tower_gap`), not a notebook-local habit, so every future driver inherits the fix automatically.

## Section 6 - Verdict and what happens next

**Closed this session, at all three epsilon dials now (bond dimension - notebook 8; truncation
algorithm and cutoff - here):** the entanglement barrier / near-exceptional point is a property of
the transfer matrix's closing gap, not of any numerical resolution knob. Phase rigidity collapses
at the identical rate whether truncating by RTM, RDM, or a tighter cutoff, and whether the bond
dimension is 32 or 64. RDM buys no practical advantage - it agrees with RTM on every eigenvalue but
costs roughly 4x more (up to ~11x at large T), so production stays RTM.

**Also closed:** the naive rank-based eigenvalue selector is unsafe near any near-degenerate
cluster, whether that cluster is triggered by frustration (NB5, larger p) or by a missing warm
start (here, cold T ladder). The fix - phase classification (`classify_tower`/`tower_gap`) plus
continuity-based selection (`pick_phys_continuity`) - is now promoted into
`src/transverse_tools.jl`, along with `block_transfer_eigs_adaptive`, which automatically escalates
from k=4 to a warm-seeded k=6 whenever a block turns out to hold no tower member besides lambda0.

**In flight:** a corrected, warm-started, checkpointed rerun of all three cluster arms
(`cluster/wall_scan_cluster.jl` v2) is running on BSC as of this writing. It uses the exact same
promoted selector and adaptive solver, and checkpoints its converged blocks to disk after every T
so a SLURM walltime kill resumes truly warm on resubmission - not just continuity-anchored. Once
that data lands, this notebook should be rerun against `results/data/cluster/warm_sweep.jld2`: the
dome/wall-location comparisons in Sections 2 and 4 are the ones that will change (the eigenvalue
and rigidity story already established here is not expected to move).

**Staged for later** (see `CLAUDE.md` §18's research-plan list): whether k=6 ever needs a further
k=8 escalation at larger p; making the solver's own output ordering continuity-stable (removing the
need for an external selector entirely); and generalizing block-warm-start persistence into an
opt-in library feature rather than a cluster-driver-only mechanism.